## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

* -0.31
* -0.02
* 0.12
* 0.44

## Loading the data

We pull the lesson pages from the course repository, the same way as in
homework 1. We pin to commit `8c1834d` so everyone works with the same
data.

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
```

Each document is a dictionary with a `filename` and `content`, and there
are 72 pages.

In [1]:
from embedder import Embedder

model = Embedder()

In [2]:
q = "How does approximate nearest neighbor search work?"
q_v = model.encode(q)

In [5]:
print(q_v[0])

-0.02058203437252893


## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

* 0.07
* 0.37
* 0.68
* 0.92

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"

for doc in documents:
    if doc["filename"] == target:
        kb = doc["content"]
        break

In [8]:
kb_v = model.encode(kb)

In [9]:
q_v.dot(kb_v)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```python
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's `content` with `encode_batch`, stack the vectors
into a matrix `X`, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its `filename`)?

* `02-vector-search/lessons/03-embeddings-dataset.md`
* `02-vector-search/lessons/06-rag-vector.md`
* `02-vector-search/lessons/07-sqlitesearch-vector.md`
* `02-vector-search/lessons/09-onnx-embedder.md`

In [10]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
# 1. Get the text from chunks
texts = [chunk["content"] for chunk in chunks]

# 2. Embed all chunks at once
X = model.encode_batch(texts)

# 3. Compute similarity with the query vector
scores = X.dot(q_v)

In [13]:
import numpy as np

best = np.argmax(scores)

print(scores[best])
print(chunks[best]["filename"])

0.6489017718578813
02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not
what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following
query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* `02-vector-search/lessons/04-vector-search.md`
* `04-evaluation/lessons/05-search-metrics.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `05-monitoring/lessons/04-metrics.md`

In [14]:
from minsearch import VectorSearch

index = VectorSearch()
index.fit(X, chunks)

In [15]:
query = "What metric do we use to evaluate a search engine?"
v = model.encode(query)

results = index.search(v, num_results=5)

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index
the same chunks with `Index` from minsearch. Use `content` as a
text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the
vector results but not in the text results?

* `02-vector-search/lessons/01-intro.md`
* `02-vector-search/lessons/02-embeddings.md`
* `02-vector-search/lessons/08-pgvector.md`
* `03-orchestration/lessons/05-rag.md`

In [16]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=[]
)

text_index.fit(chunks)

In [17]:
query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query,
    num_results=5
)

In [18]:
v = model.encode(query)

vector_results = index.search(
    v,
    num_results=5
)

In [19]:
text_files = {doc["filename"] for doc in text_results}
vector_files = {doc["filename"] for doc in vector_results}

print("Text results")
for r in text_results:
    print(r["filename"])

print("\nVector results")
for r in vector_results:
    print(r["filename"])


print("\nOnly in vector:")
print(vector_files - text_files)

Text results
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Vector results
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

Only in vector:
{'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector
search matches by meaning, so it finds relevant pages even when they use
words different from the query. But it can miss exact terms like names,
codes, or rare keywords. Text search is the opposite: it nails exact words
but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their
results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them
into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores
the raw scores from each method, which live on different scales and aren't
directly comparable. Instead, it looks only at the position of each
document in each list.

Every document scores by its position (`rank`, starting at 0) in each
list, and we sum the scores across lists with a constant `k = 60`:

```text
RRF(d) = sum over lists of  1 / (k + rank(d))
```

"Sum over lists" means we go through every ranked list and, for each list
where the document appears, add its `1 / (k + rank)` contribution. A
document found by both searches collects a score from each list, while one
found by only a single search collects just one.

The constant `k` controls how much the exact rank matters. A larger `k`
flattens the gap between positions, so the difference between rank 0 and
rank 5 counts for less. A smaller `k` does the opposite: it sharpens that
gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default.
You rarely need to tune it. Lower it when only the top results matter.
Raise it to reward documents that appear across many lists, even when they
never quite reach the top.

A document that ranks well in both lists ends up higher than one that's
only strong in a single list.

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Now run the query `"How do I give the model access to tools?"`
with vector and text search and fuse the results with `rrf`:

```python
results = rrf([vector_results, text_results])
```

Which file is ranked first after RRF?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `01-agentic-rag/lessons/16-other-frameworks.md`

Notice that this file isn't first in either search on its own - it wins
because it ranks high in both.

In [20]:
query = "How do I give the model access to tools?"

# Vector search
query_vector = model.encode(query)
vector_results = index.search(query_vector, num_results=5)

# Text search
text_results = text_index.search(query, num_results=5)

In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [22]:
results = rrf([vector_results, text_results])

print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
